# 검색 만들어보기 (1) — 어휘 검색

청크 **8개**로 검색을 처음부터 만든다. 본 코드는 1,646개라 뭐가 일어나는지 볼 수가 없다.

**셀을 위에서부터 하나씩 실행한다.** `___` 가 있는 셀은 직접 채운다.

주피터에서 셀 실행은 `Shift + Enter`.


## 0. 데이터 보기


In [1]:
import json, os

# 노트북을 learn/ 안에서 열든 레포 루트에서 열든 둘 다 되게 한다.
PATH = 'mini_chunks.jsonl' if os.path.exists('mini_chunks.jsonl') else 'learn/mini_chunks.jsonl'

CHUNKS = [json.loads(l) for l in open(PATH, encoding='utf-8')]
print(f'{len(CHUNKS)}개 불러옴  ({PATH})\n')

for i, c in enumerate(CHUNKS, 1):
    print(f"{i}. [{c['doc_id']}] {c['location'].split(' > ', 1)[-1][:45]}")
    print('   ', ' '.join(c['text'].split())[:70])


FileNotFoundError: [Errno 2] No such file or directory: 'learn/mini_chunks.jsonl'

### 연습에 쓸 질문 셋

| | 질문 | 정답 | 무엇을 드러내나 |
|---|---|---|---|
| Q1 | `유학생이 알바하려면 뭘 내야 해?` | 3번 | 말이 다르다 (알바 ≠ 시간제취업) |
| Q2 | `D-2 체류자격 변경 제출서류` | 5번 | 흔한 말 (제출서류가 5·7번에 다 있음) |
| Q3 | `시간제취업 대상` | 1번 | 말이 같다 |


In [ ]:
QUESTIONS = [
    ('Q1', '유학생이 알바하려면 뭘 내야 해?', 3),
    ('Q2', 'D-2 체류자격 변경 제출서류', 5),
    ('Q3', '시간제취업 대상', 1),
]


---
## 1단계 — 가장 단순한 검색

질문을 띄어쓰기로 자르고, 각 조각이 본문에 있으면 1점.

**빈칸**: 질문 `q` 를 띄어쓰기로 자른다. 문자열에는 `.split()` 이 붙는다.

```python
'가 나 다'.split()   # -> ['가', '나', '다']
```


In [ ]:
def naive_score(q, chunk):
    words = ___                     # q 를 띄어쓰기로 자른다
    return sum(1 for w in words if w in chunk['text'])


for qid, q, answer in QUESTIONS:
    print(f'\n{qid}  "{q}"   정답 {answer}번')
    print('  자른 결과:', q.split())
    scored = sorted(((naive_score(q, c), i) for i, c in enumerate(CHUNKS, 1)), reverse=True)
    for s, i in scored[:3]:
        mark = '  <-- 정답' if i == answer else ''
        print(f"    {s}점  {i}번  {CHUNKS[i-1]['location'].split(' > ',1)[-1][:38]}{mark}")


### 읽는 법

- **Q1 은 전부 0점.** `유학생이` 는 본문의 `유학생` 과 다른 글자다. 조사 `이` 때문이다.
  그리고 `알바` 는 매뉴얼에 아예 없다 (매뉴얼은 `시간제취업`).
- **Q2 는 맞았지만 동점이다.** 순서 운이지 실력이 아니다.
- **Q3 은 정답이 3위 안에도 없다.** 0점 아니면 1점뿐이라 변별이 안 된다.

여기서 할 일이 정해진다.

| 문제 | 해결 |
|---|---|
| 조사가 붙어서 못 찾음 | 2단계 토큰화 |
| 점수가 0/1뿐 | 3단계 TF |
| 흔한 단어와 드문 단어가 동급 | 3단계 IDF |
| 말 자체가 다름 | 4단계 임베딩 (다음 노트북) |


---
## 2단계 — 토큰화

조사와 어미를 떼는 건 형태소 분석기가 한다. 한국어용으로 Kiwi 를 쓴다.


In [ ]:
from kiwipiepy import Kiwi

k = Kiwi()
S1 = '유학생이 알바하려면 뭘 내야 해?'
S2 = 'E-7 비자를 연장하려면'


**빈칸**: `k.tokenize(S1)` 은 토큰 객체들의 리스트를 준다.
토큰 `t` 에서 글자만 꺼내려면 `t.form` 을 쓴다.


In [ ]:
print('띄어쓰기 ', S1.split())
print('형태소   ', [___ for t in k.tokenize(S1)])     # t 에서 글자만


`유학생이` 가 `유학생` + `이` 로 갈라졌을 것이다. 이제 본문의 `유학생` 과 맞출 수 있다.

**빈칸**: 토큰에는 `t.tag` 도 있다. 품사 이름이다. `(글자, 품사)` 쌍으로 보자.


In [ ]:
for t in k.tokenize(S1):
    print(f'  {t.form:10} {___}')      # 품사

print('''
  NNG 일반명사   NNP 고유명사   VV 동사   VA 형용사
  JKS/JKO/JX 조사   EC/EF 어미   SL 외국어   SN 숫자   SO 기호
''')


조사(`JKS`)와 어미(`EC`)는 **모든 문장에 나온다.** 검색에 도움이 안 되니 버린다.

**빈칸**: `t.tag` 가 `KEEP` 중 하나로 시작하는지 본다. `문자열.startswith(튜플)` 이 그 일을 한다.


In [ ]:
KEEP = ('NN', 'VV', 'VA', 'SL', 'SN')

tokens = [t.form for t in k.tokenize(S1) if ___]      # 남길 품사만
print(tokens)


### 비자 코드가 부서진다

먼저 보호 없이 `E-7` 을 잘라보자.


In [ ]:
print([ (t.form, t.tag) for t in k.tokenize(S2) ][:5])


`E` / `-` / `7` 로 쪼개진다. 숫자 `7` 은 `제7조`, `7일` 과도 겹쳐서 엉뚱한 문서가 걸린다.

그래서 **형태소 분석 전에** 정규식으로 코드를 빼낸다.

```python
re.findall(패턴, 문자열)    # 패턴에 맞는 부분들을 리스트로
re.sub(패턴, ' ', 문자열)   # 패턴에 맞는 부분을 ' ' 로 바꾼 문자열
```

**빈칸 3개**를 채운다.


In [ ]:
import re

VISA = r'[A-Za-z]-\d+(?:-\d+)?'

codes = [c.upper() for c in ___]        # S2 에서 VISA 에 맞는 것 찾기
rest  = ___                              # S2 에서 VISA 를 ' ' 로 지우기

print('빼낸 코드 ', codes)
print('남은 문장 ', rest)
print('최종      ', codes + [t.form for t in k.tokenize(rest) if t.tag.startswith(KEEP)])


이게 `src/tokenizer.py` 에 직접 쓰신 그 로직이다.
